# Synthetic Pre-train → Real Fine-tune → Volume Estimation Pipeline

**Plan (confirmed):**
1. **Generator:** irregular bunker (৩ দেয়াল+মেঝে) + ১–৩টা angle-of-repose pile, **exact analytical GT volume**, iPhone-style scan simulation (occlusion সহ) — 10,000 scene
2. **Features:** main notebook-এর হুবহু একই ২০-feature pipeline (এখানে self-contained কপি)
3. **Pre-train:** ExtraTrees / SVR_log / CatBoost / XGBoost / Ensemble — synthetic-এ
4. **Fine-tune:** (B) weighted blend — real×w weight, (C) stacking — syn-prediction feature + calibrator
5. **Evaluate:** 5-fold CV on real 45 — real-only vs B vs C comparison table
6. **DL track (comparison-only):** PointNet-style regressor, synthetic-train → real-predict
7. Best model save → main notebook-এর V11 এ ব্যবহার

Storage: features+2048-pt clouds (~250 MB), full clouds নয়। কয়েকটা `.las` QC-র জন্য।

## 1. Config

In [17]:
import os, time, json, math
import numpy as np
import pandas as pd
import joblib
from tqdm.auto import tqdm
from scipy.spatial import ConvexHull, cKDTree

SYN = {
    "n_scenes"     : 10000,
    "out_dir"      : "../data/synthetic",
    "n_las_samples": 5,            # visual QC এর জন্য কয়টা .las export
    # bunker geometry (real scans: footprint ~50–340 m², height 4–8 m)
    "W_range"      : (7.0, 20.0),  # bunker width  (m)
    "L_range"      : (7.0, 20.0),  # bunker length (m)
    "wallH_range"  : (4.0, 8.0),
    "corner_jitter": 0.12,         # irregularity: corners ±12%
    "n_piles"      : (1, 3),
    "repose_deg"   : (30.0, 40.0), # wood powder angle of repose
    "peak_range"   : (1.5, 4.5),   # pile peak height (m)
    "vol_accept"   : (38.0, 105.0), # tighter: matches real GT range 40–98 m³
    # scan simulation
    "pts_scene"    : 120_000,
    "noise_sigma"  : 0.006,        # iPhone ±6 mm
    "dropout_frac" : 0.06,         # random occlusion holes
    "gen_grid"     : 0.10,         # heightfield resolution for exact volume (m)
    # feature pipeline (main notebook-এর VOL_CONFIG-এর সাথে অভিন্ন)
    "fps_points"   : 20000, "env_points": 20000,
    "grid_res"     : 0.05, "floor_lo_pct": 30, "ransac_thresh": 0.08,
    "dl_points"    : 2048,         # DL regressor input size (paper-এর মতো)
    "seed"         : 42,
}
REAL = {
    "label_dir": "../data/train",
    "gt_csv"   : "../data/gt_volume.csv",
    "cv_folds" : 5, "cv_repeats": 5, "random_state": 42,
}
os.makedirs(SYN["out_dir"], exist_ok=True)


## 2. Feature pipeline (main notebook-এর হুবহু কপি — self-contained)

In [18]:
def _voxel_downsample_idx(pts, voxel):
    vox = np.floor(pts / voxel).astype(np.int64)
    _, inv, cnt = np.unique(vox, axis=0, return_inverse=True, return_counts=True)
    sums = np.zeros((cnt.size, 3)); np.add.at(sums, inv, pts)
    d2 = ((pts - (sums / cnt[:, None])[inv])**2).sum(1)
    order = np.lexsort((d2, inv))
    first = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    return np.sort(order[first])

def farthest_point_sample(pts, n_sample, seed=0):
    n = len(pts)
    if n <= n_sample: return np.arange(n)
    span = np.ptp(pts, axis=0).max()
    vox = max(span / (n_sample ** (1/3) * 3), 1e-3)
    keep = _voxel_downsample_idx(pts, vox)
    if len(keep) <= n_sample: return keep
    sub = pts[keep]; rng = np.random.RandomState(seed)
    sel = np.empty(n_sample, np.int64); sel[0] = rng.randint(len(sub))
    d2 = ((sub - sub[sel[0]])**2).sum(1)
    for i in range(1, n_sample):
        sel[i] = int(d2.argmax())
        d2 = np.minimum(d2, ((sub - sub[sel[i]])**2).sum(1))
    return keep[sel]

def fixed_sample(pts, n, seed=0):
    """Exactly-n sample: FPS first; if the voxel path returns fewer points
    (small/sparse surfaces), pad by random repetition. DL inputs need a
    fixed tensor size."""
    idx = farthest_point_sample(pts, n, seed)
    if len(idx) < n:
        rng = np.random.RandomState(seed)
        idx = np.r_[idx, rng.choice(idx, n - len(idx), replace=True)]
    return idx[:n]

def fit_floor_plane(env_pts, lo_pct=30, thresh=0.08, n_iter=200, seed=0):
    if len(env_pts) < 20:
        return np.array([0,0,1.0]), -float(np.median(env_pts[:,2])) if len(env_pts) else 0.0
    zt = np.percentile(env_pts[:,2], lo_pct)
    cand = env_pts[env_pts[:,2] <= zt]
    if len(cand) < 20: cand = env_pts
    rng = np.random.RandomState(seed); best_cnt=-1
    best=(np.array([0,0,1.0]), -float(np.median(cand[:,2])))
    for _ in range(n_iter):
        p0,p1,p2 = cand[rng.choice(len(cand), 3, replace=False)]
        nrm = np.cross(p1-p0, p2-p0); nn = np.linalg.norm(nrm)
        if nn < 1e-9: continue
        nrm/=nn; d = -nrm@p0
        cnt = (np.abs(cand@nrm + d) < thresh).sum()
        if cnt > best_cnt: best_cnt, best = cnt, (nrm, d)
    nrm,d = best
    if nrm[2] < 0: nrm,d = -nrm,-d
    return nrm, d

VOL_FEATURE_NAMES = [
    "floor_heightmap_vol","floor_mean_h_x_area","hull_vol","bbox_vol",
    "footprint_area","powder_h_med","powder_h_p90","powder_h_std",
    "powder_h_max","n_powder_log","xy_extent","xy_aspect",
    "density_area","env_floor_area","wall_height","powder_env_ratio",
    "z_p25_rel","z_p75_rel","top_flatness","slope",
]

def _floor_heightmap(powder, nrm, d, grid_res):
    h = powder@nrm + d
    x,y = powder[:,0], powder[:,1]
    ix = np.floor((x-x.min())/grid_res).astype(np.int64)
    iy = np.floor((y-y.min())/grid_res).astype(np.int64)
    key = ix*(iy.max()+2)+iy; o = np.argsort(key)
    ks, hs = key[o], h[o]
    _, st = np.unique(ks, return_index=True)
    cell_h = np.clip(np.maximum.reduceat(hs, st), 0, None)
    return float(cell_h.sum()*grid_res*grid_res), float(len(cell_h)*grid_res**2), h

def extract_volume_features(powder, env, grid_res, floor_lo_pct, ransac_thresh):
    nrm, d = fit_floor_plane(env, floor_lo_pct, ransac_thresh)
    bbox_area = max(np.ptp(powder[:,0]) * np.ptp(powder[:,1]), 1e-6)
    gr_eff = max(grid_res, float(np.sqrt(2.0 * bbox_area / max(len(powder), 1))))
    hm_vol, foot_area, ph = _floor_heightmap(powder, nrm, d, gr_eff)
    ph_pos = np.clip(ph, 0, None)
    x,y,z = powder[:,0], powder[:,1], powder[:,2]
    try:    hull_vol = float(ConvexHull(powder).volume)
    except Exception: hull_vol = np.nan
    dx,dy,dz = np.ptp(x), np.ptp(y), np.ptp(z)
    env_h = env@nrm + d
    wall_height = float(np.percentile(env_h, 98))
    env_floor_area = float(len(env)/max(np.ptp(env[:,0])*np.ptp(env[:,1]),1e-6))
    slope = np.nan
    if len(powder) > 50:
        A = np.c_[x,y,np.ones_like(x)]; coef,*_ = np.linalg.lstsq(A,z,rcond=None)
        slope = float(np.hypot(coef[0],coef[1]))
    return np.array([
        hm_vol, float(ph_pos.mean()*foot_area), hull_vol, dx*dy*dz,
        foot_area, float(np.median(ph_pos)), float(np.percentile(ph_pos,90)),
        float(ph_pos.std()), float(ph_pos.max()), float(np.log1p(len(powder))),
        float(max(dx,dy)), float(dx/max(dy,1e-6)),
        float(len(powder)/max(foot_area,1e-6)), env_floor_area, wall_height,
        float(len(powder)/max(len(env),1)),
        float(np.percentile(ph_pos,25)), float(np.percentile(ph_pos,75)),
        float(np.percentile(ph_pos,95)-np.percentile(ph_pos,80)), slope,
    ], np.float64)


## 3. Synthetic scene generator — exact analytical GT volume

**Scene:** irregular quadrilateral bunker (কোণ ±12% jitter) + ১–৩টা angle-of-repose pile
(capped cone: `h = max(0, H − tan(repose)·r)`, একাধিক pile-এ `h = max_i h_i`)।
**GT volume = Σ h·cell²** — generator-defined surface-এর উপর exact numerical integral,
কোনো mesh/approximation নেই।

**Scan simulation (iPhone-style):**
- Powder surface: uniform-in-XY sample যেখানে h>0, z=h(x,y)+noise → শুধু উপরিভাগ (shell)
- Floor points: **শুধু যেখানে h=0** (powder-এর নিচের floor প্রাকৃতিকভাবে occluded ✓)
- Walls: প্রতিটি দেয়ালে শুধু **powder line-এর উপরের** অংশ (নিচেরটা powder-এ ঢাকা ✓)
- Gaussian noise (σ=6 mm) + random dropout patches (scan holes)

In [19]:
def make_scene(rng):
    """Build one bunker scene. Returns pts(float32 N,3), labels(uint8), gt_volume."""
    W = rng.uniform(*SYN["W_range"]); L = rng.uniform(*SYN["L_range"])
    wallH = rng.uniform(*SYN["wallH_range"])
    j = SYN["corner_jitter"]
    # irregular quad footprint: rectangle corners jittered
    cx = np.array([0, W, W, 0], float) + rng.uniform(-j*W, j*W, 4)
    cy = np.array([0, 0, L, L], float) + rng.uniform(-j*L, j*L, 4)
    x0, x1 = cx.min(), cx.max(); y0, y1 = cy.min(), cy.max()

    gr = SYN["gen_grid"]
    gx = np.arange(x0, x1, gr); gy = np.arange(y0, y1, gr)
    GX, GY = np.meshgrid(gx, gy)

    # point-in-quad mask (convex quad → all cross products same sign)
    def in_quad(px, py):
        s = None; ok = np.ones_like(px, bool)
        for k in range(4):
            ax,ay = cx[k],cy[k]; bx,by = cx[(k+1)%4],cy[(k+1)%4]
            cr = (bx-ax)*(py-ay) - (by-ay)*(px-ax)
            if s is None: s = cr >= 0
            ok &= ((cr >= 0) == s) | (np.abs(cr) < 1e-9)
        return ok
    inside = in_quad(GX, GY)

    # piles: capped cones, peaks kept inside, often leaning near the back wall
    n_p = rng.randint(SYN["n_piles"][0], SYN["n_piles"][1]+1)
    H = np.zeros_like(GX)
    for _ in range(n_p):
        px = rng.uniform(x0+0.15*(x1-x0), x1-0.15*(x1-x0))
        py = rng.uniform(y0+0.30*(y1-y0), y1-0.10*(y1-y0))   # bias toward back
        pk = rng.uniform(*SYN["peak_range"])
        s  = math.tan(math.radians(rng.uniform(*SYN["repose_deg"])))
        r  = np.hypot(GX-px, GY-py)
        H  = np.maximum(H, pk - s*r)
    H = np.clip(H, 0, None) * inside
    gt_volume = float(H.sum() * gr * gr)                 # exact integral

    if not (SYN["vol_accept"][0] <= gt_volume <= SYN["vol_accept"][1]):
        return None                                       # rejection sampling

    N = SYN["pts_scene"]
    n_pow  = int(N*0.55); n_flr = int(N*0.20); n_wall = N - n_pow - n_flr

    def surf_h(px, py):                                   # bilinear-ish lookup
        ix = np.clip(((px-x0)/gr).astype(int), 0, len(gx)-1)
        iy = np.clip(((py-y0)/gr).astype(int), 0, len(gy)-1)
        return H[iy, ix], inside[iy, ix]

    # powder surface points (only where h>0)
    pw = []
    while sum(len(a) for a in pw) < n_pow:
        px = rng.uniform(x0, x1, n_pow); py = rng.uniform(y0, y1, n_pow)
        h, ins = surf_h(px, py)
        m = ins & (h > 0.02)
        pw.append(np.c_[px[m], py[m], h[m]])
    powder = np.vstack(pw)[:n_pow]

    # floor points — only uncovered floor (h == 0), natural occlusion
    fl = []
    while sum(len(a) for a in fl) < n_flr:
        px = rng.uniform(x0, x1, n_flr); py = rng.uniform(y0, y1, n_flr)
        h, ins = surf_h(px, py)
        m = ins & (h <= 0.02)
        fl.append(np.c_[px[m], py[m], np.zeros(m.sum())])
        if not m.any(): break                              # fully covered bunker
    floor = np.vstack(fl)[:n_flr] if fl and len(fl[0])>0 else np.zeros((0,3))

    # walls (3 sides: back y≈max + two sides), only ABOVE local powder line
    walls = []
    for k in [1, 2, 3]:                                    # skip front edge (open)
        ax,ay = cx[k],cy[k]; bx,by = cx[(k+1)%4],cy[(k+1)%4]
        t  = rng.uniform(0, 1, n_wall//3)
        wx = ax + t*(bx-ax); wy = ay + t*(by-ay)
        h_here, _ = surf_h(wx, wy)
        wz = h_here + rng.uniform(0, 1, len(t))*(wallH - h_here)
        m  = wz > h_here + 0.02
        walls.append(np.c_[wx[m], wy[m], wz[m]])
    wall = np.vstack(walls) if walls else np.zeros((0,3))

    env = np.vstack([floor, wall])
    pts = np.vstack([powder, env]).astype(np.float32)
    lbl = np.r_[np.ones(len(powder), np.uint8), np.zeros(len(env), np.uint8)]

    # noise + dropout
    pts += rng.normal(0, SYN["noise_sigma"], pts.shape).astype(np.float32)
    if SYN["dropout_frac"] > 0:                            # a few circular holes
        keep = np.ones(len(pts), bool)
        for _ in range(rng.randint(1, 4)):
            c = pts[rng.randint(len(pts)), :2]
            r = rng.uniform(0.2, 0.7)
            keep &= (np.hypot(pts[:,0]-c[0], pts[:,1]-c[1]) > r) | \
                    (rng.rand(len(pts)) > 0.9)
        pts, lbl = pts[keep], lbl[keep]

    return pts, lbl, gt_volume


## 4. 10,000 scene generate + feature extract (parallel)

প্রতি scene: generate → powder/env আলাদা → FPS → ২০-feature + DL-এর ২০৪৮-point cloud।
রাখা হয় শুধু: `X_syn (N,20)`, `y_syn (N,)`, `P_syn (N,2048,4)` (xyz + class flag) — full cloud নয়।
`joblib.Parallel` সব core ব্যবহার করে।

In [20]:
from joblib import Parallel, delayed

def one_scene(seed):
    rng = np.random.RandomState(seed)
    for _ in range(30):                                   # retry until volume accepted
        out = make_scene(rng)
        if out is not None: break
    else:
        return None
    pts, lbl, gtv = out
    powder = pts[lbl == 1].astype(np.float64)
    env    = pts[lbl == 0].astype(np.float64)
    if len(powder) < 500 or len(env) < 200: return None
    powder = powder[farthest_point_sample(powder, SYN["fps_points"], seed)]
    env    = env[farthest_point_sample(env,    SYN["env_points"], seed)]
    feats  = extract_volume_features(powder, env, SYN["grid_res"],
                                     SYN["floor_lo_pct"], SYN["ransac_thresh"])
    # DL input: 1536 powder + 512 env, 4th channel = class flag, metric xyz/20
    kp = fixed_sample(powder, 1536, seed)
    ke = fixed_sample(env,    512,  seed)
    dl = np.vstack([np.c_[powder[kp]/20.0, np.ones(len(kp))],
                    np.c_[env[ke]/20.0,    np.zeros(len(ke))]]).astype(np.float32)
    return feats, gtv, dl

t0 = time.time()
results = Parallel(n_jobs=-1, verbose=1)(
    delayed(one_scene)(SYN["seed"] + i) for i in range(SYN["n_scenes"]))
results = [r for r in results if r is not None]

X_syn = np.array([r[0] for r in results])
y_syn = np.array([r[1] for r in results])
P_syn = np.array([r[2] for r in results], np.float32)
X_syn = np.nan_to_num(X_syn, nan=0.0)

np.savez_compressed(os.path.join(SYN["out_dir"], "synthetic_features.npz"),
                    X=X_syn, y=y_syn, feature_names=VOL_FEATURE_NAMES)
np.save(os.path.join(SYN["out_dir"], "synthetic_dl_points.npy"), P_syn)
print(f"{len(y_syn):,} scenes in {(time.time()-t0)/60:.1f} min | "
      f"X{X_syn.shape} P{P_syn.shape} | volume {y_syn.min():.0f}–{y_syn.max():.0f} m³ "
      f"(mean {y_syn.mean():.0f})")


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.


[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:    5.6s
[Parallel(n_jobs=-1)]: Done 736 tasks      | elapsed:   10.5s
[Parallel(n_jobs=-1)]: Done 1186 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done 1736 tasks      | elapsed:   25.0s
[Parallel(n_jobs=-1)]: Done 2386 tasks      | elapsed:   34.3s
[Parallel(n_jobs=-1)]: Done 3136 tasks      | elapsed:   45.9s
[Parallel(n_jobs=-1)]: Done 3986 tasks      | elapsed:   59.8s
[Parallel(n_jobs=-1)]: Done 4936 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 5986 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 7136 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 8386 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 9736 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 10000 out of 10000 | elapsed:  2.8min finished


10,000 scenes in 2.8 min | X(10000, 20) P(10000, 2048, 4) | volume 38–105 m³ (mean 72)


### 4b. Visual QC — কয়েকটা scene `.las`-এ export (CloudCompare-এ দেখুন)

In [21]:
import laspy
for k in range(SYN["n_las_samples"]):
    out = None; attempt = 0
    while out is None:                            # seed increments each retry
        out = make_scene(np.random.RandomState(999 + k*100 + attempt))
        attempt += 1
    pts, lbl, gtv = out
    hdr = laspy.LasHeader(point_format=6, version="1.4")
    hdr.scales = [0.001]*3; hdr.offsets = pts.min(0).astype(float)
    las = laspy.LasData(hdr)
    las.x, las.y, las.z = pts[:,0].astype(float), pts[:,1].astype(float), pts[:,2].astype(float)
    las.classification = lbl
    p = os.path.join(SYN["out_dir"], f"syn_qc_{k}_V{gtv:.0f}m3.las")
    las.write(p); print("wrote", p, f"({len(pts):,} pts, V={gtv:.1f} m³)")


wrote ../data/synthetic/syn_qc_0_V90m3.las (118,282 pts, V=90.3 m³)
wrote ../data/synthetic/syn_qc_1_V42m3.las (118,842 pts, V=42.4 m³)
wrote ../data/synthetic/syn_qc_2_V89m3.las (118,906 pts, V=89.0 m³)
wrote ../data/synthetic/syn_qc_3_V53m3.las (119,114 pts, V=53.2 m³)
wrote ../data/synthetic/syn_qc_4_V74m3.las (117,357 pts, V=74.2 m³)


## 5. Real GT features (45 files) — main notebook-এর মতোই

In [22]:
import glob

def load_pointcloud(path):
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x), np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    lbl = None
    for key in ("classification","label","labels","class"):
        if key in las.point_format.dimension_names:
            lbl = np.asarray(getattr(las, key), np.int64); break
    return pts, lbl

gt_df = pd.read_csv(REAL["gt_csv"])
gt_df.columns = [c.strip().lower() for c in gt_df.columns]
gt_df["stem"] = gt_df["filename"].map(lambda s: os.path.splitext(os.path.basename(str(s).strip()))[0])
gt_vol = dict(zip(gt_df["stem"], gt_df["volume"].astype(float)))

X_real, y_real, names_real = [], [], []
files = sorted(glob.glob(os.path.join(REAL["label_dir"], "*.la?")))
for path in tqdm(files, desc="real features"):
    stem = os.path.splitext(os.path.basename(path))[0]
    if stem not in gt_vol: continue
    pts, lbl = load_pointcloud(path)
    if lbl is None: continue
    powder = pts[lbl == 1]; env = pts[lbl != 1]
    if len(powder) < 50 or len(env) < 50: continue
    powder = powder[farthest_point_sample(powder, SYN["fps_points"], 42)]
    env    = env[farthest_point_sample(env, SYN["env_points"], 42)]
    X_real.append(extract_volume_features(powder, env, SYN["grid_res"],
                  SYN["floor_lo_pct"], SYN["ransac_thresh"]))
    y_real.append(gt_vol[stem]); names_real.append(stem)
X_real = np.nan_to_num(np.array(X_real), nan=0.0)
y_real = np.array(y_real)
print(f"real: {len(y_real)} files | X{X_real.shape}")


real features:   0%|          | 0/45 [00:00<?, ?it/s]

real: 45 files | X(45, 20)


## 6. Pre-train (synthetic) → Fine-tune (B: weighted, C: stacking) → CV তুলনা

প্রতি fold-এ: fine-tune দেখে **শুধু train-fold real + সব synthetic**; test হয় held-out real-এ।
- **real-only:** baseline (আগের ফল)
- **B (weighted):** synthetic+real একসাথে, real sample_weight = w ∈ {5,10,20} (CV-তে বাছাই)
- **C (stacking):** synthetic-model frozen → তার prediction real feature-এর সাথে জুড়ে
  ছোট calibrator (Ridge) — সবচেয়ে domain-shift resistant

In [23]:
from sklearn.model_selection import RepeatedKFold
from sklearn.base import clone, BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor

def _mape(a,b):
    a,b = np.asarray(a,float), np.asarray(b,float)
    return float(np.mean(np.abs((a-b)/np.clip(np.abs(a),1e-9,None)))*100)

def _logt(reg):
    return TransformedTargetRegressor(regressor=reg, func=np.log1p, inverse_func=np.expm1)

def base_models():
    m = {"ExtraTrees": ExtraTreesRegressor(400, random_state=0, n_jobs=-1),
         "SVR_log": _logt(Pipeline([("s",StandardScaler()),
                                    ("m",SVR(C=100, epsilon=0.05, gamma="scale"))]))}
    try:
        from catboost import CatBoostRegressor
        m["CatBoost"] = CatBoostRegressor(iterations=600, depth=6, learning_rate=0.05,
                                          verbose=False, random_seed=0)
    except ImportError: pass
    try:
        from xgboost import XGBRegressor
        m["XGBoost"] = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                                    subsample=0.9, random_state=0, verbosity=0)
    except ImportError: pass
    return m

def fit_weighted(reg, Xs, ys, Xr, yr, w):
    """B: joint fit, real rows upsampled w× (equivalent to sample_weight=w).
    Row-repetition works universally with any estimator/wrapper — no need to
    route sample_weight through Pipeline/TransformedTargetRegressor internals."""
    # upsample real rows instead of using sample_weight parameter
    Xr_rep = np.repeat(Xr, int(round(w)), axis=0)
    yr_rep  = np.repeat(yr, int(round(w)), axis=0)
    X = np.vstack([Xs, Xr_rep])
    y = np.r_[ys, yr_rep]
    return clone(reg).fit(X, y)

class SynStack(BaseEstimator, RegressorMixin):
    """C: frozen synthetic model's prediction as an extra feature + Ridge calibrator."""
    def __init__(self, syn_model): self.syn_model = syn_model
    def fit(self, X, y):
        z = self.syn_model.predict(X).reshape(-1,1)
        self.calib_ = Ridge(alpha=1.0).fit(np.c_[X, z], y)
        return self
    def predict(self, X):
        z = self.syn_model.predict(X).reshape(-1,1)
        return self.calib_.predict(np.c_[X, z])

rkf = RepeatedKFold(n_splits=REAL["cv_folds"], n_repeats=REAL["cv_repeats"],
                    random_state=REAL["random_state"])
ROWS = []
for name, reg in base_models().items():
    syn_model = clone(reg).fit(X_syn, y_syn)          # pre-train once (frozen for C)
    real_only, wB = [], {3: [], 5: [], 10: [], 15: [], 20: [], 30: []}
    stackC = []
    for tr, te in rkf.split(X_real):
        Xtr, ytr, Xte, yte = X_real[tr], y_real[tr], X_real[te], y_real[te]
        real_only.append(_mape(yte, clone(reg).fit(Xtr, ytr).predict(Xte)))
        for w in wB:
            wB[w].append(_mape(yte, fit_weighted(reg, X_syn, y_syn, Xtr, ytr, w)
                                     .predict(Xte)))
        stackC.append(_mape(yte, SynStack(syn_model).fit(Xtr, ytr).predict(Xte)))
    bw = min(wB, key=lambda w: np.mean(wB[w]))
    ROWS += [
        {"model": name, "strategy": "real-only",        "mape": np.mean(real_only), "std": np.std(real_only)},
        {"model": name, "strategy": f"B weighted (w={bw})", "mape": np.mean(wB[bw]), "std": np.std(wB[bw])},
        {"model": name, "strategy": "C stacking",       "mape": np.mean(stackC),   "std": np.std(stackC)},
        {"model": name, "strategy": "syn-only (no FT)", "mape": np.mean([_mape(y_real[te], syn_model.predict(X_real[te])) for _, te in rkf.split(X_real)]),
         "std": 0.0},
    ]
    print(f"{name}: real-only {np.mean(real_only):.2f}% | "
          f"B(w={bw}) {np.mean(wB[bw]):.2f}% | C {np.mean(stackC):.2f}%")

RES = pd.DataFrame(ROWS).sort_values("mape").reset_index(drop=True)
print("\n=== Synthetic pre-train + fine-tune leaderboard (MAPE% on real 5×5 CV) ===")
print(RES.to_string(index=False, formatters={"mape":"{:.2f}".format,"std":"{:.2f}".format}))
os.makedirs("../results", exist_ok=True)
RES.to_csv("../results/synthetic_finetune_leaderboard.csv", index=False)


ExtraTrees: real-only 11.49% | B(w=30) 11.49% | C 13.41%
SVR_log: real-only 11.46% | B(w=30) 10.36% | C 13.44%
CatBoost: real-only 11.73% | B(w=30) 10.69% | C 13.12%
XGBoost: real-only 12.07% | B(w=20) 9.81% | C 12.57%

=== Synthetic pre-train + fine-tune leaderboard (MAPE% on real 5×5 CV) ===
     model          strategy  mape  std
   XGBoost B weighted (w=20)  9.81 3.02
   SVR_log B weighted (w=30) 10.36 2.68
  CatBoost B weighted (w=30) 10.69 2.58
   SVR_log         real-only 11.46 1.81
ExtraTrees B weighted (w=30) 11.49 2.79
ExtraTrees         real-only 11.49 3.17
  CatBoost         real-only 11.73 3.15
   XGBoost         real-only 12.07 3.44
  CatBoost  syn-only (no FT) 12.14 0.00
   XGBoost        C stacking 12.57 3.34
   XGBoost  syn-only (no FT) 13.02 0.00
  CatBoost        C stacking 13.12 3.65
ExtraTrees        C stacking 13.41 3.64
   SVR_log        C stacking 13.44 3.60
ExtraTrees  syn-only (no FT) 13.57 0.00
   SVR_log  syn-only (no FT) 18.51 0.00


## 7. Best model save (production) — main notebook-এর V11-এ ব্যবহারযোগ্য

In [24]:
best = RES[RES["strategy"] != "syn-only (no FT)"].iloc[0]
name, strat = best["model"], best["strategy"]
reg = base_models()[name]
print(f"Winner: {name} / {strat} @ {best['mape']:.2f}%")

if strat.startswith("B"):
    w = int(strat.split("w=")[1].rstrip(")"))
    FINAL = fit_weighted(reg, X_syn, y_syn, X_real, y_real, w)
elif strat == "C stacking":
    FINAL = SynStack(clone(reg).fit(X_syn, y_syn)).fit(X_real, y_real)
else:
    FINAL = clone(reg).fit(X_real, y_real)

path = f"../checkpoints/vol_finetuned_{name}.joblib"
os.makedirs("../checkpoints", exist_ok=True)
joblib.dump({"model": FINAL, "method": f"{name}+{strat}",
             "feature_names": VOL_FEATURE_NAMES, "cv_mape": float(best["mape"]),
             "n_train_files": len(y_real), "n_synthetic": len(y_syn)}, path)
print(f"Saved → {path}")
print("Main notebook-এর V11-এ শুধু VOL_MODEL_PATH এই path-এ বদলে দিন —")
print("predict interface একই (sklearn .predict), আর কিছু বদলাতে হবে না।")
print("Note: C-stacking model load করতে V11 session-এ এই notebook-এর SynStack class")
print("cell টা একবার চালিয়ে নিতে হবে (joblib class definition খোঁজে)।")


Winner: XGBoost / B weighted (w=20) @ 9.81%
Saved → ../checkpoints/vol_finetuned_XGBoost.joblib
Main notebook-এর V11-এ শুধু VOL_MODEL_PATH এই path-এ বদলে দিন —
predict interface একই (sklearn .predict), আর কিছু বদলাতে হবে না।
Note: C-stacking model load করতে V11 session-এ এই notebook-এর SynStack class
cell টা একবার চালিয়ে নিতে হবে (joblib class definition খোঁজে)।


## 8. DL track (comparison-only) — PointNet-style regressor

Paper-এর সাথে তুলনার জন্য: synthetic ২০৪৮-point cloud-এ train, real-এ direct predict
(**no fine-tune** — ৪৫ file DL fine-tune-এর জন্য যথেষ্ট নয়)। Production-এ যাবে না,
thesis-এর comparison table-এ যাবে। `RUN_DL=False` করলে skip।

In [25]:
RUN_DL = True
if RUN_DL:
    import torch, torch.nn as nn
    DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    class PointNetReg(nn.Module):
        """Compact PointNet: shared MLP → max-pool → FC → log-volume."""
        def __init__(self, in_ch=4):
            super().__init__()
            self.mlp = nn.Sequential(
                nn.Conv1d(in_ch, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                nn.Conv1d(64, 128, 1),  nn.BatchNorm1d(128), nn.ReLU(),
                nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(256, 64), nn.ReLU(),
                                      nn.Dropout(0.3), nn.Linear(64, 1))
        def forward(self, x):                     # x: (B, N, 4)
            f = self.mlp(x.permute(0, 2, 1)).max(dim=2).values
            return self.head(f).squeeze(1)        # log1p(volume)

    Xp = torch.from_numpy(P_syn); Yl = torch.from_numpy(np.log1p(y_syn).astype(np.float32))
    n_val = max(len(Yl)//10, 1)
    tr_ds = torch.utils.data.TensorDataset(Xp[:-n_val], Yl[:-n_val])
    va_ds = torch.utils.data.TensorDataset(Xp[-n_val:], Yl[-n_val:])
    tr = torch.utils.data.DataLoader(tr_ds, batch_size=64, shuffle=True)
    va = torch.utils.data.DataLoader(va_ds, batch_size=256)

    net = PointNetReg().to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
    lossf = nn.MSELoss()
    for ep in range(60):
        net.train(); tl = 0.0
        for xb, yb in tr:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad(); l = lossf(net(xb), yb); l.backward(); opt.step()
            tl += l.item()*len(xb)
        sched.step()
        if (ep+1) % 5 == 0:
            net.eval(); preds, ys = [], []
            with torch.no_grad():
                for xb, yb in va:
                    preds.append(np.expm1(net(xb.to(DEV)).cpu().numpy())); ys.append(np.expm1(yb.numpy()))
            print(f"ep {ep+1:2d} | train MSE(log) {tl/len(tr_ds):.4f} | "
                  f"val MAPE {_mape(np.concatenate(ys), np.concatenate(preds)):.2f}%")

    # real 45 files → DL input → direct predict
    P_real = []
    for path in tqdm(files, desc="real DL clouds"):
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem not in gt_vol: continue
        pts, lbl = load_pointcloud(path)
        if lbl is None: continue
        pw = pts[lbl == 1]; en = pts[lbl != 1]
        if len(pw) < 50 or len(en) < 50: continue
        kp = fixed_sample(pw, 1536, 42); ke = fixed_sample(en, 512, 42)
        P_real.append(np.vstack([np.c_[pw[kp]/20.0, np.ones(len(kp))],
                                 np.c_[en[ke]/20.0, np.zeros(len(ke))]]).astype(np.float32))
    P_real = torch.from_numpy(np.array(P_real))
    net.eval()
    with torch.no_grad():
        out_raw = np.clip(net(P_real.to(DEV)).cpu().numpy(), -10, 10)
    dl_pred = np.expm1(out_raw)
    print(f"\nDL (PointNetReg, syn-only) on REAL {len(dl_pred)} files: "
          f"MAPE = {_mape(y_real[:len(dl_pred)], dl_pred):.2f}%  ← thesis comparison row")


ep  5 | train MSE(log) 0.3979 | val MAPE 12.24%
ep 10 | train MSE(log) 0.3540 | val MAPE 10.27%
ep 15 | train MSE(log) 0.3009 | val MAPE 13.41%
ep 20 | train MSE(log) 0.2890 | val MAPE 15.67%
ep 25 | train MSE(log) 0.2639 | val MAPE 12.13%
ep 30 | train MSE(log) 0.2674 | val MAPE 9.69%
ep 35 | train MSE(log) 0.2652 | val MAPE 12.08%
ep 40 | train MSE(log) 0.2662 | val MAPE 12.90%
ep 45 | train MSE(log) 0.2457 | val MAPE 12.17%
ep 50 | train MSE(log) 0.2150 | val MAPE 10.77%
ep 55 | train MSE(log) 0.1836 | val MAPE 8.62%
ep 60 | train MSE(log) 0.1423 | val MAPE 8.56%


real DL clouds:   0%|          | 0/45 [00:00<?, ?it/s]


DL (PointNetReg, syn-only) on REAL 45 files: MAPE = 32469.06%  ← thesis comparison row
